In [1]:
import os
print(os.getcwd())

C:\Users\user\Documents\AskMyDB


In [16]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("API_KEY")
print("Key loaded!" if api_key else "Key missing")

Key loaded!


In [17]:
from google import genai

client = genai.Client(api_key=api_key)
print("Package found successfully")

Package found successfully


In [18]:
import mysql.connector

# Connect to your MySQL database
conn = mysql.connector.connect(
    host="localhost",
    user="root",           # your MySQL username
    password="Your_Password",  # your MySQL password
    database="bank_churn"
)

cursor = conn.cursor()

# Get the table structure so Gemini knows what columns exist
cursor.execute("DESCRIBE customers")
schema = cursor.fetchall()

print("Table structure:")
for col in schema:
    print(col)

Table structure:
('customer_id', 'int', 'YES', '', None, '')
('credit_score', 'int', 'YES', '', None, '')
('country', 'text', 'YES', '', None, '')
('gender', 'text', 'YES', '', None, '')
('age', 'int', 'YES', '', None, '')
('tenure', 'int', 'YES', '', None, '')
('balance', 'double', 'YES', '', None, '')
('products_number', 'int', 'YES', '', None, '')
('credit_card', 'int', 'YES', '', None, '')
('active_member', 'int', 'YES', '', None, '')
('estimated_salary', 'double', 'YES', '', None, '')
('churn', 'int', 'YES', '', None, '')


In [19]:
import time

def ask_database(question, max_retries=3):
    schema_description = """
    Table name: customers
    Columns:
    - customer_id (int)
    - credit_score (int)
    - country (text) - values: France, Germany, Spain
    - gender (text)
    - age (int)
    - tenure (int)
    - balance (double)
    - products_number (int)
    - credit_card (int)
    - active_member (int)
    - estimated_salary (double)
    - churn (int) - 1 = churned, 0 = stayed
    """
    
    sql_prompt = f"""You are a MySQL expert. Given this table structure:
{schema_description}

Write a single MySQL query to answer this question: "{question}"

Return ONLY the raw SQL query, ending with a semicolon. No explanation, no markdown."""

    for attempt in range(max_retries):
        try:
            sql_response = client.models.generate_content(
                model="gemini-flash-lite-latest",
                contents=sql_prompt
            )
            generated_sql = sql_response.text.strip().replace("```sql", "").replace("```", "").strip()
            
            print(f"Generated SQL: {generated_sql}\n")
            
            cursor.execute(generated_sql)
            result = cursor.fetchall()
            columns = [desc[0] for desc in cursor.description]
            
            explain_prompt = f"""The user asked: "{question}"
The SQL result: {result}
Columns: {columns}

Explain this in one clear sentence for a non-technical person, including the actual numbers."""

            explanation = client.models.generate_content(
                model="gemini-flash-lite-latest",
                contents=explain_prompt
            )
            
            print(f"Answer: {explanation.text}")
            return generated_sql, result, explanation.text
            
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                print("Retrying in 5 seconds...")
                time.sleep(5)
            else:
                print("All retries failed. Try again in a moment.")

In [21]:
print("=" * 50)
print("AskMyDB - Ask questions about your bank customer data")
print("Type 'exit' to quit")
print("=" * 50)

while True:
    user_question = input("\nYour question: ")

    if user_question.lower() in ['exit', 'quit']:
        print("Goodbye!")
        break

    try:
        ask_database(user_question)
    except Exception as e:
        print(f"Something went wrong: {e}")
        print("Try rephrasing your question.")

AskMyDB - Ask questions about your bank customer data
Type 'exit' to quit



Your question:  What is the overall churn rate?


Generated SQL: SELECT AVG(churn) AS churn_rate FROM customers;

Answer: The overall churn rate is 20.37%, meaning that about one in every five customers has stopped using the service.



Your question:  Which country has the highest churn rate?


Generated SQL: SELECT country FROM customers GROUP BY country ORDER BY AVG(churn) DESC LIMIT 1;

Answer: Germany has the highest churn rate, with a total of 814 customers leaving.


...

Generated SQL: SELECT 
    active_member, 
    AVG(churn) AS churn_rate, 
    COUNT(*) AS total_customers, 
    SUM(churn) AS churned_customers 
FROM customers 
GROUP BY active_member;

Answer: No, contrary to what you might expect, inactive members actually churn less at 14.3% (735 out of 5,151 customers) compared to active members who churn at a higher rate of 26.9% (1,302 out of 4,849 customers).



Your question:  Is there a difference in balance between customers who churned and those who stayed?


Generated SQL: SELECT churn, AVG(balance) AS average_balance, MIN(balance) AS min_balance, MAX(balance) AS max_balance, STDDEV(balance) AS stddev_balance FROM customers GROUP BY churn;

Answer: On average, customers who churned had a higher balance of $91,108 compared to $72,745 for those who stayed.



Your question:  exit


Goodbye!
